# 🎬 VibeMV Ultimate: Stable Video Diffusion (Updated)

## ⭐ Highest Quality Option - With Drive Integration & Checkpointing

## Workflow
1. Mount Google Drive & load storyboard
2. Generate high-quality SDXL keyframes (saved to Drive)
3. Animate each keyframe with SVD (saved to Drive)
4. Stitch into final video with audio

**Result:** Professional music video quality! 🎬

In [ ]:
# @title ✅ Check GPU
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {vram_gb:.1f} GB')
    if vram_gb < 14:
        print('   ⚠️  Will use aggressive memory optimizations')
else:
    print('❌ No GPU!')
    raise SystemExit

In [ ]:
# @title 💾 Mount Google Drive & Load Timeline
from google.colab import drive
import json
import os

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/vibemv_output'
os.makedirs(DRIVE_PATH, exist_ok=True)

timeline_path = '/content/drive/MyDrive/timeline.json'

with open(timeline_path, 'r') as f:
    timeline = json.load(f)

is_vibeframe = 'video_prompt' in timeline['scenes'][0]
print(f"✅ Loaded {len(timeline['scenes'])} scenes")
print(f"   Duration: {timeline.get('audio_duration', 'N/A')}s\n")

for i, scene in enumerate(timeline['scenes'][:3]):
    if is_vibeframe:
        print(f"  {i+1}. {scene.get('description', '')[:60]}...")

In [ ]:
# @title 🛠️ Install SVD Dependencies (4-5 minutes)
%%capture

!pip install -q torch torchvision
!pip install -q diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg opencv-python pillow
!pip install -q xformers

print('✅ SVD ready!')

In [ ]:
# @title ⚙️ Configure Batch Processing
from typing import List

SCENES_PER_BATCH = 10
timeline['scenes'] = [s for s in timeline['scenes'] if s.get('duration', 0) > 0]

print(f"Total scenes: {len(timeline['scenes'])}")
print(f"Batches needed: {(len(timeline['scenes']) + SCENES_PER_BATCH - 1) // SCENES_PER_BATCH}")

---

## Step 1: Generate Keyframes

In [ ]:
# @title 🎨 Generate SDXL Keyframes (Batch Mode)
from diffusers import StableDiffusionXLPipeline
import torch
import shutil
import gc

os.makedirs('keyframes', exist_ok=True)

print('Loading SDXL...')
sdxl_pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16'
).to('cuda')

sdxl_pipe.enable_vae_slicing()

NEGATIVE = 'ugly, blurry, low quality, distorted'
all_keyframes = []

for batch_num in range(0, len(timeline['scenes']), SCENES_PER_BATCH):
    batch = timeline['scenes'][batch_num:batch_num + SCENES_PER_BATCH]
    print(f"\n🎨 Batch {batch_num // SCENES_PER_BATCH + 1}: {len(batch)} scenes\n")
    
    for i, scene in enumerate(batch):
        global_idx = batch_num + i
        prompt = scene.get('video_prompt', scene.get('prompt', scene.get('description', '')))
        duration = scene.get('duration', 4.0)
        
        print(f"Keyframe {global_idx+1}: {prompt[:70]}...")
        
        image = sdxl_pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE,
            num_inference_steps=40,
            guidance_scale=8.5,
            height=576,
            width=1024
        ).images[0]
        
        img_path = f"keyframes/scene_{global_idx:03d}.png"
        image.save(img_path)
        all_keyframes.append({'path': img_path, 'duration': duration})
        print(f"  ✅ Saved\n")
        
        if (i + 1) % 3 == 0:
            torch.cuda.empty_cache()
    
    batch_drive_path = f"{DRIVE_PATH}/keyframes_batch{batch_num // SCENES_PER_BATCH + 1}"
    shutil.copytree('keyframes', batch_drive_path, dirs_exist_ok=True)
    print(f"✅ Batch backed up to Drive: {batch_drive_path}\n")
    gc.collect()

del sdxl_pipe
torch.cuda.empty_cache()
gc.collect()

shutil.copytree('keyframes', f"{DRIVE_PATH}/all_keyframes", dirs_exist_ok=True)
print(f"\n✅ {len(all_keyframes)} keyframes ready and backed up!")

---

## Step 2: Animate with SVD

In [ ]:
# @title 🎬 Animate with Stable Video Diffusion (Batch Mode)
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
import torch
import gc
import os

print('Loading Stable Video Diffusion...')
svd_pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16'
)
svd_pipe.enable_model_cpu_offload()
svd_pipe.unet.enable_forward_chunking()

os.makedirs('animated_clips', exist_ok=True)
all_clips = []

for batch_num in range(0, len(all_keyframes), SCENES_PER_BATCH):
    batch = all_keyframes[batch_num:batch_num + SCENES_PER_BATCH]
    print(f"\n🎬 Batch {batch_num // SCENES_PER_BATCH + 1}: {len(batch)} scenes\n")
    
    for i, keyframe in enumerate(batch):
        global_idx = batch_num + i
        print(f"Animating scene {global_idx+1}/{len(all_keyframes)}...")
        
        image = load_image(keyframe['path'])
        image = image.resize((1024, 576))
        
        torch.cuda.empty_cache()
        gc.collect()
        
        frames = svd_pipe(
            image,
            num_frames=50,
            decode_chunk_size=2,
            num_inference_steps=25,
            motion_bucket_id=127,
            fps=25
        ).frames[0]
        
        clip_path = f"animated_clips/scene_{global_idx:03d}.mp4"
        export_to_video(frames, clip_path, fps=25)
        all_clips.append({'path': clip_path, 'duration': keyframe['duration']})
        
        print(f"  ✅ {len(frames)} frames\n")
        
        torch.cuda.empty_cache()
        gc.collect()
    
    batch_drive_path = f"{DRIVE_PATH}/clips_batch{batch_num // SCENES_PER_BATCH + 1}"
    shutil.copytree('animated_clips', batch_drive_path, dirs_exist_ok=True)
    print(f"✅ Batch backed up to Drive: {batch_drive_path}\n")

del svd_pipe
torch.cuda.empty_cache()
gc.collect()

shutil.copytree('animated_clips', f"{DRIVE_PATH}/all_clips", dirs_exist_ok=True)
print(f"\n✅ All {len(all_clips)} clips animated and backed up!")

---

## Step 3: Add Audio & Stitch

In [ ]:
# @title 🎵 Upload Audio File
from google.colab import files

print("Upload your audio file (MP3, WAV, or MP4):")
uploaded = files.upload()
audio_filename = list(uploaded.keys())[0]
print(f"✅ Audio loaded: {audio_filename}")

In [ ]:
# @title 🎬 Stitch Final Video with Audio
from moviepy.editor import VideoFileClip, concatenate_videoclips, AudioFileClip

print('🎬 Assembling final video...\n')

video_clips = []
for i, clip in enumerate(all_clips):
    print(f"Loading clip {i+1}...")
    vc = VideoFileClip(clip['path'])
    
    target = clip['duration']
    if vc.duration < target:
        loops = int(target / vc.duration) + 1
        vc = vc.loop(n=loops).set_duration(target)
    else:
        vc = vc.set_duration(target)
    
    video_clips.append(vc)

print('\n🎞️ Concatenating clips...')
final = concatenate_videoclips(video_clips, method='compose')

print('🎵 Adding audio...')
audio = AudioFileClip(audio_filename)
final = final.set_audio(audio)

print('\n💾 Exporting...')
output_path = 'vibemv_svd_ultimate.mp4'
final.write_videofile(
    output_path,
    fps=25,
    codec='libx264',
    preset='medium',
    audio_codec='aac'
)

for vc in video_clips:
    vc.close()
final.close()
audio.close()

shutil.copy(output_path, f"{DRIVE_PATH}/{output_path}")
print(f'\n✅ Ultimate quality MV complete!')
print(f'📁 Saved to Drive: {DRIVE_PATH}/{output_path}')

files.download(output_path)

---

## 🔄 Resume from Checkpoint

In [ ]:
# @title 🔄 Load Previously Generated Keyframes from Drive
import shutil
import os

keyframes_dir = f"{DRIVE_PATH}/all_keyframes"
if os.path.exists(keyframes_dir):
    shutil.copytree(keyframes_dir, 'keyframes', dirs_exist_ok=True)
    
    from PIL import Image
    import json
    
    all_keyframes = []
    for i, scene in enumerate(timeline['scenes']):
        img_path = f"keyframes/scene_{i:03d}.png"
        if os.path.exists(img_path):
            all_keyframes.append({
                'path': img_path,
                'duration': scene.get('duration', 4.0)
            })
    
    print(f"✅ Loaded {len(all_keyframes)} keyframes from Drive")
    print(f"→ Ready for SVD animation step!")
else:
    print("❌ No keyframes found in Drive. Run keyframe generation first.")

In [ ]:
# @title 🔄 Load Previously Animated Clips from Drive
import shutil
import os

clips_dir = f"{DRIVE_PATH}/all_clips"
if os.path.exists(clips_dir):
    shutil.copytree(clips_dir, 'animated_clips', dirs_exist_ok=True)
    
    all_clips = []
    for i, keyframe in enumerate(all_keyframes):
        clip_path = f"animated_clips/scene_{i:03d}.mp4"
        if os.path.exists(clip_path):
            all_clips.append({
                'path': clip_path,
                'duration': keyframe['duration']
            })
    
    print(f"✅ Loaded {len(all_clips)} clips from Drive")
    print(f"→ Ready for final stitch step!")
else:
    print("❌ No clips found in Drive. Run SVD animation first.")

---

## ✅ Done!

**Key improvements:**
- ✅ Google Drive integration - no lost work on timeout
- ✅ Batch processing - handles long timelines safely
- ✅ Checkpointing - resume from any step
- ✅ Fixed dependency conflicts
- ✅ Longer SVD clips (50 frames) for smoother motion
- ✅ Audio integration
- ✅ Aggressive VRAM cleanup
- ✅ Progressive backups

**vs Original:**
- No lost progress ✅
- Handles 47+ scenes ✅
- Smoother video ✅
- Audio included ✅
- Resume capability ✅